this notebook contains the euclidian analysis of the crime data

# 1.0 Load data

In [ ]:
# load the packages
import pandas as pd
import json
import math
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import geopandas as gpd
import scipy.stats as stats
from shapely.geometry import Point
from matplotlib.patches import Patch
from sklearn.neighbors import NearestNeighbors
from statsmodels.stats.multitest import multipletests
from statsmodels.discrete.discrete_model import NegativeBinomial

In [ ]:
import gc
gc.collect()

In [ ]:
# load the cleaned boundary data
bradford_boundary = gpd.read_file('bradford_boundary.geojson', use_arrow=True)

In [ ]:
# load the data
df = gpd.read_file('crime_nearest_school.geojson', use_arrow=True)

# 2.0 Preliminary analysis

In [ ]:
# total crimes on school days and holidays
crime_totals = (
    df.groupby('term_time_type')
    .size())

crime_totals

In [ ]:
# calculate the percentages
crime_percentage = crime_totals / crime_totals.sum() * 100
crime_percentage.round(4)

In [ ]:
# number of days in each period
day_totals = (
    df[['date_only', 'term_time_type']]
    .drop_duplicates()
    .groupby('term_time_type')
    .size())

day_totals

In [ ]:
# average number of crimes per day
daily_rate = crime_totals / day_totals
daily_rate

In [ ]:
# calculate excess
percent_higher = (
    (daily_rate.loc['Term time'] - daily_rate.loc['holiday'])
    / daily_rate.loc['holiday']
) * 100

percent_higher

School day rates are 15% higher than non-school day rates

# 3.0 Buffer separation

In [ ]:
# filter for each buffer
within_250 = df[df['distance_to_school_m'] <= 250].copy()
over_250 = df[df['distance_to_school_m'] > 250].copy()
within_500 = df[df['distance_to_school_m'] <= 500].copy()
over_500 = df[df['distance_to_school_m'] > 500].copy()
within_750 = df[df['distance_to_school_m'] <= 750].copy()
over_750 = df[df['distance_to_school_m'] > 750].copy()
within_1000 = df[df['distance_to_school_m'] <= 1000].copy()
over_1000 = df[df['distance_to_school_m'] > 1000].copy()

In [ ]:
# check that data is well filtered: replace the distances
print(over_1000['distance_to_school_m'].min())
print((over_1000['distance_to_school_m'] <= 1000).sum())

In [ ]:
# tract the tables
within_250['buffer_track'] = 'within_250'
over_250['buffer_track'] = 'over_250'

within_500['buffer_track'] = 'within_500'
over_500['buffer_track'] = 'over_500'

within_750['buffer_track'] = 'within_750'
over_750['buffer_track'] = 'over_750'

within_1000['buffer_track'] = 'within_1000'
over_1000['buffer_track'] = 'over_1000'

In [ ]:
# concatenate
buffer_df = pd.concat(
    [within_250, over_250, within_500, over_500,
     within_750, over_750, within_1000, over_1000],
    ignore_index=True
)

In [ ]:
buffer_df['buffer_track'].value_counts()

In [ ]:
# extract the numerical buffer distance
buffer_df['distance'] = (
    buffer_df['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

# identify within and further-than observations
buffer_df['location'] = (
    buffer_df['buffer_track']
    .str.split('_')
    .str[0]
    .replace({
        'within': 'Within',
        'over': 'Further than'
    })
)

# 4.0 Global analysis

## 4.1 How many crimes per week?

In [ ]:
# group each buffer by the termtime
per_week_totals = (buffer_df.groupby(['buffer_track', 'distance', 'location', 'year&week', 'term_time_type'])
                .size()
                .unstack(fill_value=0))

In [ ]:
# rename the columns
per_week_totals = (per_week_totals.rename(
    columns={'Term time': 'totalcrime_schoolday',
             'holiday': 'totalcrime_holiday'})
                .reset_index())

## 4.2 What is the crime rate per week?

Get the number of days per term time and merge to the crime totals

In [ ]:
# Get the number of days per term time type using the initial data, df
days_per_week = df[['year&week', 'day_name', 'term_time_type']].drop_duplicates()

In [ ]:
# get the number of days
n_termdays = (
    days_per_week[days_per_week['term_time_type'] == 'Term time']
    .groupby('year&week')
    .size()
    .rename('n_schooldays')
    .reset_index())

In [ ]:
# get the number of days
n_holidays = (
    days_per_week[days_per_week['term_time_type'] == 'holiday']
    .groupby('year&week')
    .size()
    .rename('n_holidays')
    .reset_index())

In [ ]:
# merge n_term_days and n_holidays
number_of_days = n_termdays.merge(
    n_holidays,
    on='year&week',
    how='outer'
).fillna(0)

In [ ]:
# merge the crime totals to the number of days
days_per_week = per_week_totals.merge(
    number_of_days,
    on="year&week",
    how="left"
)

In [ ]:
# calculate the weekly crime rates
# Calculate crimes per school day and per holiday day in each week
days_per_week["mean_crime_schoolday"] = (
    days_per_week["totalcrime_schoolday"] / days_per_week["n_schooldays"]
)

days_per_week["mean_crime_holiday"] = (
    days_per_week["totalcrime_holiday"] / days_per_week["n_holidays"]
)

In [ ]:
# replace the infinities
days_per_week[['mean_crime_schoolday', 'mean_crime_holiday']] = days_per_week[
    ['mean_crime_schoolday', 'mean_crime_holiday']].replace([np.inf, -np.inf], np.nan)

plot the table

In [ ]:
# set the tables
buffer_order = buffer_df['buffer_track'].unique()

In [ ]:
# set the layout
fig, axes = plt.subplots(
    nrows=8, ncols=1, figsize=(16, 40),
    sharex=False, sharey=False)

axes = axes.flatten()

# loop all the tables
for ax, buffer_name in zip(axes, buffer_order):
    # filter the buffer
    plot_data = (days_per_week[days_per_week['buffer_track'] == buffer_name]
        .sort_values('year&week').reset_index(drop=True))

    # use numerical positions to control the x-axis labels
    x = np.arange(len(plot_data))

    # layer the school day rate
    ax.plot(x, plot_data['mean_crime_schoolday'],
            label='Term time', marker='o', ms=3, linewidth=1.5)

    ax.plot(x, plot_data['mean_crime_holiday'],
            label='Holiday', marker='o', ms=3, linewidth=1.5)

    # label each chart
    ax.set_title(f'Crime Rate per Week at {buffer_name.replace('_',' ').title()} meters')
    ax.set_xlabel('Year and week')
    ax.set_ylabel('Mean crimes per day')
    ax.grid(alpha=0.3)
    ax.legend(loc='best')

    # show eight x-axis labels
    step = max(1, len(plot_data) // 8)
    tick_positions = x[::step]

    ax.set_xticks(tick_positions)
    ax.set_xticklabels(plot_data['year&week'].iloc[::step],
                       rotation=0, ha='center')

# add title
fig.suptitle('Weekly crime rates by walk time', fontsize=18, y=1.01)
plt.tight_layout()
plt.show()

Gaps indicate weeks containing no days in the relevant term-time category; therefore, a crime rate could not be calculated.

## 4.3 What is the pattern of occurence of crime types by week

In [ ]:
# count crimetypes
crime_type = (buffer_df
              .groupby(['buffer_track', 'year&week', 'offence', 'term_time_type'])
              .size()
              .unstack('term_time_type', fill_value=0)
              .reindex(columns=['Term time', 'holiday'], fill_value=0))

In [ ]:
# rename the columns
crime_type = crime_type.rename(columns={
    'Term time': 'schoolday',
    'holiday': 'holiday'})

In [ ]:
crime_type = crime_type.unstack('offence', fill_value=0)

In [ ]:
# flatten the names
crime_type.columns = [
    f"n_{str(offence).strip().lower().replace('-','').replace(' ', '_')}_{term_type}"
    for term_type, offence in crime_type.columns]

# reset index
crime_type = crime_type.reset_index()

In [ ]:
crime_type_df = days_per_week.merge(crime_type, on=['buffer_track', 'year&week'], how='left')

In [ ]:
# reorder the table
crime_type_df = crime_type_df[[
    'buffer_track', 'year&week', 'totalcrime_schoolday', 'totalcrime_holiday', 'n_schooldays', 'n_holidays',
    'mean_crime_schoolday', 'mean_crime_holiday',
    
    'n_arson_schoolday', 'n_arson_holiday',
    'n_bicycle_theft_schoolday', 'n_bicycle_theft_holiday',
    'n_burglary__business_and_community_schoolday', 'n_burglary__business_and_community_holiday',
    'n_burglary__residential_schoolday', 'n_burglary__residential_holiday', 
    'n_criminal_damage_schoolday', 'n_criminal_damage_holiday',
    'n_miscellaneous_crimes_against_society_schoolday', 'n_miscellaneous_crimes_against_society_holiday',
    'n_other_sexual_offences_schoolday', 'n_other_sexual_offences_holiday',
    'n_other_theft_schoolday', 'n_other_theft_holiday',
    'n_possession_of_drugs_schoolday',  'n_possession_of_drugs_holiday',
    'n_possession_of_weapons_schoolday', 'n_possession_of_weapons_holiday',
    'n_public_order_offences_schoolday', 'n_public_order_offences_holiday',
    'n_rape_schoolday', 'n_rape_holiday',
    'n_robbery_of_business_property_schoolday', 'n_robbery_of_business_property_holiday',
    'n_robbery_of_personal_property_schoolday', 'n_robbery_of_personal_property_holiday',
    'n_shoplifting_schoolday', 'n_shoplifting_holiday',
    'n_theft_from_motor_vehicle_schoolday', 'n_theft_from_motor_vehicle_holiday',
    'n_theft_from_the_person_schoolday', 'n_theft_from_the_person_holiday',
    'n_theft_of_motor_vehicle_schoolday', 'n_theft_of_motor_vehicle_holiday',
    'n_trafficking_in_controlled_drugs_schoolday', 'n_trafficking_in_controlled_drugs_holiday',
    'n_vehicle_interference_schoolday', 'n_vehicle_interference_holiday',
    'n_violence_with_injury_schoolday', 'n_violence_with_injury_holiday',
    'n_violence_without_injury_schoolday', 'n_violence_without_injury_holiday'
]]

In [ ]:
# save the summary table
crime_type_df.to_csv('euclidian_crimetype_weekly_summary.csv', index=False)

Sense check: confirm any mismatches

In [ ]:
# confirm there is no mismatch
schoolday_cols = [c for c in crime_type_df.columns
                  if c.startswith('n_') and c.endswith('_schoolday')]

holiday_cols = [c for c in crime_type_df.columns
                if c.startswith('n_') and c.endswith('_holiday')]

In [ ]:
check_schoolday = crime_type_df[schoolday_cols].sum(axis=1)
check_holiday = crime_type_df[holiday_cols].sum(axis=1)

In [ ]:
mismatch_schoolday = crime_type_df[check_schoolday != crime_type_df['totalcrime_schoolday']]
mismatch_holiday = crime_type_df[check_holiday != crime_type_df['totalcrime_holiday']]

print("Mismatched school day rows:", len(mismatch_schoolday))
print("Mismatched holiday rows:", len(mismatch_holiday))

No mismatches

## 4.4 How is school day rate compared to the holiday rate?: What is the school-to-holiday rate ratio?

In [ ]:
# get the taotal number of days
n_schooldays = day_totals['Term time']
n_holidays = day_totals['holiday']

In [ ]:
buffers = crime_type_df['buffer_track'].unique()

In [ ]:
# calculate rate for each buffer
buffer_list = []

# loop for each buffer
for buffer in buffers:
    # filter the buffer
    buffer_data = crime_type_df[crime_type_df['buffer_track'] == buffer]

    # School-day rate
    schoolday_rate = buffer_data['totalcrime_schoolday'].sum() / n_schooldays
    
    # Holiday rate
    holiday_rate = buffer_data['totalcrime_holiday'].sum() / n_holidays

    # Rate ratio and percentage difference
    rate_ratio = schoolday_rate / holiday_rate
    percent_difference = (
        (schoolday_rate - holiday_rate)
        / holiday_rate
    ) * 100

    # append the rates
    buffer_list.append({
        'buffer_track': buffer,
        'schoolday_rate': schoolday_rate,
        'holiday_rate': holiday_rate,
        'rate_ratio': rate_ratio,
        'percent_difference': percent_difference
    })

In [ ]:
# put into a table
rate_comparison = pd.DataFrame(buffer_list)
rate_comparison

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[buffer_order, ['schoolday_rate', 'holiday_rate']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 6), width=0.8)

# add labels and titles
ax.set_title('School day and holiday crime rates by buffer', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Crime Rate (crimes per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# plot just within
# get the buffers
within_order = ['within_250', 'within_500', 'within_750', 'within_1000']
over_order = ['over_250', 'over_500', 'over_750', 'over_1000']

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[within_order, ['schoolday_rate', 'holiday_rate']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Within Buffer Comparison: School day and holiday crime rates by buffer', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Crime Rate (crimes per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
plot_data = (
    rate_comparison
    .set_index('buffer_track')
    .loc[within_order, ['schoolday_rate', 'holiday_rate']]
)

In [ ]:
rate_change = (
    rate_comparison
    .set_index('buffer_track')
    .loc[within_order, ['schoolday_rate', 'holiday_rate']]
    .pct_change()
    .mul(100)
    .dropna()
)

In [ ]:
ax = rate_change.plot(
    kind='line',
    marker='o',
    figsize=(10, 5),
    linewidth=2
)

ax.axhline(0, color='black', linestyle='--', linewidth=1)

ax.set_title(
    'Percentage Change in Crime Rate as the Buffer Expands',
    fontweight='bold'
)
ax.set_xlabel('Buffer')
ax.set_ylabel('Change from previous buffer (%)')
ax.tick_params(axis='x', rotation=0)
ax.legend(['School day', 'Holiday'])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[within_order, ['percent_difference']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Within Buffer Percent Difference Comparison', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Crime Rate (crimes per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

The percentage difference reduced as the buffer increased: dfference between school day rate and non-school day rate is highest at 250 meters, and least at 750 meters.

I will plot similar chart for the further than metrics.

In [ ]:
# set the table for further than
# obtain the school day and non school day rates
plot_data = (
    rate_comparison
    .set_index('buffer_track')
    .loc[over_order, ['schoolday_rate', 'holiday_rate']]
)

In [ ]:
# compute the percentage difference
rate_change = (
    rate_comparison
    .set_index('buffer_track')
    .loc[over_order, ['schoolday_rate', 'holiday_rate']]
    .pct_change()
    .mul(100)
    .dropna()
)

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[over_order, ['schoolday_rate', 'holiday_rate']])

In [ ]:
# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Beyond Buffer Comparison: School day and holiday crime rates by buffer', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Crime Rate (crimes per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# plot line chart
ax = rate_change.plot(
    kind='line',
    marker='o',
    figsize=(10, 5),
    linewidth=2
)

ax.axhline(0, color='black', linestyle='--', linewidth=1)

# label and titles
ax.set_title(
    'Percentage Change in Crime Rate Outside the Buffer as the Buffer Expands',
    fontweight='bold'
)
ax.set_xlabel('Buffer')
ax.set_ylabel('Change from previous buffer (%)')
ax.tick_params(axis='x', rotation=0)
ax.legend(['School day', 'Holiday'])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

school day rates are all higher than the non school day rates. Highest rates are at 250 meters, and least at 1000 meters. So the percentage difference:

In [ ]:
# prepare table for rates
plot_data = (
    rate_comparison.set_index('buffer_track')
    .loc[over_order, ['percent_difference']])

# plot table
ax = plot_data.plot(
    kind='bar', figsize=(12, 5), width=0.8)

# add labels and titles
ax.set_title('Beyond Buffer Percent Comparison', fontweight='bold')
ax.set_xlabel('Buffer')
ax.set_ylabel('Mean Crime Rate (crimes per day)')
ax.tick_params(axis='x', rotation=0)
ax.legend(title='')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

...percentage diference is highest at 1000 meters, and least at 250 meters. &50 meters and 1000 meters are almost same mangnitude.

## 4.5 Is crime effect more concentrated wthin the distance or furthan than the distance?

In [ ]:
# extract the numerical buffer distance
rate_comparison['distance'] = (
    rate_comparison['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int)
)

# identify whether the observation is within or further than
rate_comparison['location'] = (
    rate_comparison['buffer_track']
    .str.split('_')
    .str[0]
    .replace({
        'within': 'Within',
        'over': 'Further than'
    })
)

In [ ]:
# pivot the table
w2f_rate_df = (
    rate_comparison
    .pivot(index='distance',
           columns='location',
           values=['schoolday_rate', 'holiday_rate']))

In [ ]:
# flatten and rename column
w2f_rate_df.columns = [
    f'{location}_{term}'
    for term, location in w2f_rate_df.columns]

w2f_rate_df = w2f_rate_df.reset_index()

In [ ]:
# reorder table
w2f_rate_df = w2f_rate_df[
    ['distance','Within_schoolday_rate','Within_holiday_rate',
     'Further than_schoolday_rate', 'Further than_holiday_rate']]

In [ ]:
w2f_rate_df['schoolday_w2f_ratio'] = (
    w2f_rate_df['Within_schoolday_rate']
    / w2f_rate_df['Further than_schoolday_rate']
)

w2f_rate_df['holiday_w2f_ratio'] = (
    w2f_rate_df['Within_holiday_rate']
    / w2f_rate_df['Further than_holiday_rate']
)

In [ ]:
w2f_rate_df

At 250 m, the W2F ratios are very low for both school days (0.08) and holidays (0.08), indicating that crime rates within the immediate school vicinity are substantially lower than those recorded beyond the buffer. The ratio increases to approximately 0.31-0.33 at 500 m, suggesting that crime remains more concentrated outside the buffer, although the difference is less pronounced.

By 750 m, the ratios approach 1.0 (0.96 for school days and 1.07 for holidays), indicating that crime rates inside and outside the buffer become broadly similar. At 1000 m, the ratios exceed 1.0 for both school days (1.96) and holidays (2.13), showing that crime rates within the buffer are nearly twice those observed beyond the buffer.

## 4.6 For each distance comparison, do the observed school-day and holiday crime counts within and beyond the buffer differ from the counts expected based on the amount of school-day and holiday exposure?

In [ ]:
# Exposure days
total_days = n_schooldays + n_holidays

schoolday_prop = n_schooldays / total_days
holiday_prop = n_holidays / total_days

In [ ]:
# set buffers
buffer_pairs = {
    250: ('within_250', 'over_250'),
    500: ('within_500', 'over_500'),
    750: ('within_750', 'over_750'),
    1000: ('within_1000', 'over_1000')
}

In [ ]:
# aggregate from weekly crime counts
buffer_counts = (
    per_week_totals
    .groupby('buffer_track')[['totalcrime_schoolday', 'totalcrime_holiday']]
    .sum()
).reset_index()

In [ ]:
# extract walking time
buffer_counts['distance'] = (
    buffer_counts['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int))

# extract within or over
buffer_counts['location'] = (
    buffer_counts['buffer_track']
    .str.split('_')
    .str[0])

In [ ]:
# unsatck the loaction
gof_table = (
    buffer_counts
    .set_index(['distance', 'location'])
    [['totalcrime_schoolday', 'totalcrime_holiday']]
    .unstack('location'))

In [ ]:
# arrange and rename the columns
gof_table = gof_table[[
    ('totalcrime_schoolday', 'within'),
    ('totalcrime_holiday', 'within'),
    ('totalcrime_schoolday', 'over'),
    ('totalcrime_holiday', 'over')
]]

gof_table.columns = [
    'Within school day',
    'Within holiday',
    'Further school day',
    'Further holiday'
]

gof_table = (
    gof_table
    .reset_index()
    .sort_values('distance')
)

In [ ]:
# apply the goodness of fit
gof_results = []

for _, row in gof_table.iterrows():

    # Observed counts
    observed = [
        row['Within school day'],
        row['Within holiday'],
        row['Further school day'],
        row['Further holiday']]

    # Total crimes in each location
    within_total = (row['Within school day'] + row['Within holiday'])
    further_total = (row['Further school day'] + row['Further holiday'])

    # Expected counts based on exposure days
    expected = [
        within_total * schoolday_prop,
        within_total * holiday_prop,
        further_total * schoolday_prop,
        further_total * holiday_prop
    ]

    # Goodness-of-fit test
    chi2, p_value = stats.chisquare(
        f_obs=observed,
        f_exp=expected,
        ddof=1)

    gof_results.append({
        'distance': row['distance'],
        'chi_square': chi2,
        'p_value': p_value
    })

In [ ]:
buffer_test = pd.DataFrame(gof_results)
buffer_test

here is significant difference across the several buffers.

## 4.7 Are crimes disproportionately concentrated on school days within and beyond the school buffer areas?

In [ ]:
# sum counts
gof_table['Within total'] = (
    gof_table['Within school day']
    + gof_table['Within holiday']
)

gof_table['Further total'] = (
    gof_table['Further school day']
    + gof_table['Further holiday']
)

In [ ]:
# calculate expected count
gof_table['Expected within school day'] = (
    gof_table['Within total'] * schoolday_prop)

gof_table['Expected within holiday'] = (
    gof_table['Within total'] * holiday_prop)

gof_table['Expected further school day'] = (
    gof_table['Further total'] * schoolday_prop)

gof_table['Expected further holiday'] = (
    gof_table['Further total'] * holiday_prop)

In [ ]:
# compare observed with expected
gof_table['Within difference (%)'] = (
    (gof_table['Within school day'] - gof_table['Expected within school day'])
    / gof_table['Expected within school day']) * 100

gof_table['Further difference (%)'] = (
    (gof_table['Further school day'] - gof_table['Expected further school day'])
    / gof_table['Expected further school day']) * 100

In [ ]:
# view table
gof_table[[
    'distance', 'Within school day', 'Expected within school day',
    'Within difference (%)', 'Further school day', 'Expected further school day',
    'Further difference (%)'
]].round(2)

Observed crime counts exceeded expected counts both within and beyond the school buffers, indicating that crime levels were higher than expected across the study area. While the strongest within-buffer excess was observed at 250 m, the differences were modest overall and did not provide consistent evidence of increasing crime concentration closer to schools.

# 5.0 Crime types

## 5.1 How do the crime types compare?

In [ ]:
# aggregate crime by types
crime_composition = (buffer_df
                     .groupby(['buffer_track', 'term_time_type', 'offence'])
                     .size()
                     .rename('crime_count')
                     .reset_index())

In [ ]:
# compute teh total crime
crime_composition['total_crimes'] = (
    crime_composition
    .groupby(['buffer_track', 'term_time_type'])['crime_count']
    .transform('sum'))

In [ ]:
# calculate share
crime_composition['share_percent'] = (
    crime_composition['crime_count']
    / crime_composition['total_crimes']) * 100

In [ ]:
# pivot the table
composition_table = (
    crime_composition
    .pivot(index=['buffer_track', 'offence'],
           columns='term_time_type',
           values='share_percent')
    .fillna(0)
    # rename columns
    .rename(columns={
        'Term time': 'schoolday_share',
        'holiday': 'holiday_share'})
    .reset_index())

In [ ]:
# calculate difference in percentage points
composition_table['share_difference_pp'] = (
    composition_table['schoolday_share']
    - composition_table['holiday_share']
)

Visualise the compositions

In [ ]:
# get the offences
offences = sorted(composition_table['offence'].unique())

In [ ]:
# set teh colors
colours = (list(plt.cm.tab20.colors)
           + list(plt.cm.tab20b.colors))

colours = colours[:len(offences)]

In [ ]:
fig, axes = plt.subplots(
    nrows=8, ncols=1,
    figsize=(18, 34), sharex=False)

# loop for buffer order and transpose
for ax, buffer_name in zip(axes, buffer_order):
    plot_data = (
        composition_table[composition_table['buffer_track'] == buffer_name]
        .set_index('offence')
        .reindex(offences)
        [['schoolday_share', 'holiday_share']]
        .fillna(0)
        .T)
    
    plot_data.index = ['School day', 'Holiday']
    plot_data.plot(kind='barh', stacked=True, ax=ax,
                   color=colours, width=0.65, legend=False)

    # labels and titles
    ax.set_title(f'Share of Crime types at {buffer_name.replace('_', ' ').title()} meters')
    ax.set_xlim(0, 100)
    ax.set_ylabel('')
    ax.set_xlabel('Share of crimes (%)')
    ax.grid(axis='x', alpha=0.3)

# set one shared leged for the crimetypes
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, title='Crime type',
           loc='center left', bbox_to_anchor=(0.81, 0.5))

fig.suptitle('Crime-type composition on school days and holidays', fontsize=18, y=0.995)
plt.tight_layout(rect=[0, 0, 0.80, 0.98])
plt.show()

## 5.2 What is the rate of occurence of each type within the period?

In [ ]:
# count each offence by buffer and term type
offence_rate = (
    buffer_df
    .groupby(['buffer_track', 'offence', 'term_time_type'])
    .size()
    .unstack('term_time_type', fill_value=0)
    .reindex(columns=['Term time', 'holiday'], fill_value=0))

In [ ]:
# rename columns
offence_rate = offence_rate.rename(columns={
    'Term time': 'schoolday_count',
    'holiday': 'holiday_count'}).reset_index()

In [ ]:
# calculate school day rate
offence_rate['schoolday_rate'] = (offence_rate['schoolday_count']
                                  / n_schooldays)

# calculate holiday rate
offence_rate['holiday_rate'] = (offence_rate['holiday_count'] / n_holidays)

In [ ]:
# calculate teh rate ratio
offence_rate['rate_ratio'] = (
    offence_rate['schoolday_rate']
    / offence_rate['holiday_rate'])

In [ ]:
# calculate the percentage difference
offence_rate['percent_difference'] = (
    offence_rate['rate_ratio'] - 1) * 100

In [ ]:
pd.set_option('display.max_row', None)
offence_rate[['buffer_track', 'offence', 'schoolday_count', 'holiday_count',
              'schoolday_rate', 'holiday_rate', 'rate_ratio', 'percent_difference']]

visualise teh table

In [ ]:
# sort offences
offence_order = (
    offence_rate
    .groupby('offence')[['schoolday_rate', 'holiday_rate']]
    .mean()
    .mean(axis=1)
    .sort_values(ascending=False)
    .index)

In [ ]:
# create seaparte tables for each type
# create for school day rate
crimetype_schoolday_rate = (
    offence_rate
    .pivot(index='offence',
           columns='buffer_track',
           values='schoolday_rate')
    .reindex(index=offence_order, columns=buffer_order))

# create for holiday rate
crimetype_holiday_rate = (
    offence_rate
    .pivot(
        index='offence',
        columns='buffer_track',
        values='holiday_rate')
    .reindex(index=offence_order,
             columns=buffer_order))

In [ ]:
# set maximum for both charts
maximum_rate = max(
    crimetype_schoolday_rate.max().max(),
    crimetype_holiday_rate.max().max())

In [ ]:
# set teh layout
fig, axes = plt.subplots(nrows=1, ncols=2,
                         figsize=(17, 10), sharey=True)

# plot for the school day rate
sns.heatmap(crimetype_schoolday_rate, ax=axes[0],
            cmap='YlOrRd', vmin=0, vmax=maximum_rate, cbar=False)

# plot the holiday map
sns.heatmap(crimetype_holiday_rate, ax=axes[1], cmap='YlOrRd',
            vmin=0, vmax=maximum_rate, cbar_kws={'label': 'Rate (crimes per day)'})

# set labels and titles
axes[0].set_title('School day occurrence rates', fontweight='bold')
axes[1].set_title('Holiday occurrence rates', fontweight='bold')
axes[0].set_xlabel('Buffer')
axes[1].set_xlabel('Buffer')
axes[0].set_ylabel('')
axes[1].set_ylabel('')

axes[0].tick_params(axis='x', rotation=0)
axes[1].tick_params(axis='x', rotation=0)

fig.suptitle('Crime-type occurrence rates by buffer and term type', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.show()

Plot for the rate ratio

In [ ]:
# sum the counts
offence_rate['total'] = offence_rate['schoolday_count'] + offence_rate['holiday_count']

In [ ]:
# Crime types with more than 50 crimes in over_250
interest = (
    offence_rate.loc[
        (offence_rate['buffer_track'] == 'over_250')
        & (offence_rate['total'] > 50),
        'offence'
    ]
    .unique()
)

In [ ]:
# Select these crime types across all buffers
crimetype_rate_ratio = (
    offence_rate.loc[
        offence_rate['offence'].isin(interest),
        ['buffer_track', 'offence', 'rate_ratio']
    ]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=['rate_ratio'])
    .copy()
)

In [ ]:
# pivot the table
df_crimetype_rate_ratio = (
    crimetype_rate_ratio
    .pivot_table(index='offence',
                 columns='buffer_track',
                 values='rate_ratio',
                 aggfunc='first')
    .reindex(index=offence_order,
             columns=buffer_order).dropna()
)

In [ ]:
df_crimetype_rate_ratio

In [ ]:
# plot the rate ratio
plt.figure(figsize=(14, 6))
sns.heatmap(
    df_crimetype_rate_ratio,
    # mask=df_crimetype_rate_ratio.isna(),
    cmap='YlOrRd',
    center=1,
    annot=True,
    fmt='.2f'
)

plt.ylabel('')
plt.xlabel('')
plt.show()

show the comparison

In [ ]:
# set the buffers
distances = [250, 500, 750, 1000]

In [ ]:
# initialise layout
fig, axes = plt.subplots(
    nrows=4, ncols=1,
    figsize=(15, 40), sharex=False, sharey=False)

# loop for each
for ax, distance in zip(axes, distances):
    # Names used in buffer_track
    within_name = f'within_{distance}'
    over_name = f'over_{distance}'

    # Select the within and over data for this distance
    chart_data = (
        crimetype_rate_ratio[crimetype_rate_ratio['buffer_track'].isin([within_name, over_name])]
        .pivot_table(index='offence',
                     columns='buffer_track',
                     values='rate_ratio',
                     aggfunc='first')
        .reindex(columns=[within_name, over_name])
        .rename(columns={within_name: 'Within',
                         over_name: 'Over'})
        .dropna(how='all'))

    # Sort by the Within rate ratio
    chart_data = chart_data.sort_values(by='Within', ascending=True,na_position='first')

    # Plot horizontal bars
    chart_data.plot(kind='barh', ax=ax, width=0.8)

    # A ratio of 1 means equal school-day and holiday rates
    ax.axvline(x=1,color='black', linestyle='--', linewidth=1)
    ax.set_title(f'{distance} m buffer: crime-type rate ratios')
    ax.set_xlabel('School-day-to-holiday rate ratio')
    ax.set_ylabel('Crime type')

    ax.legend(title='Buffer category',loc='best')
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('Crime-type rate ratios within and beyond each buffer', fontsize=18, y=1.002)
plt.tight_layout()
plt.show()

## 5.3 How does the within crimes differ from further crimes?

In [ ]:
# copy the offence table
comparison_table = offence_rate.copy()
# comparison_table

In [ ]:
# clean the distance track column
# extract the metric measure
comparison_table['distance'] = (
    comparison_table['buffer_track']
    .str.extract(r'(\d+)')[0]
    .astype(int))

In [ ]:
# split to create within or outside category and rename the splits
comparison_table['location'] = (
    comparison_table['buffer_track']
    .str.split('_')
    .str[0]                  # use the first part of the split
    # map the names
    .map({'within': 'Within',
          'over': 'Further than'
    }))

In [ ]:
# pivot the table
comparison_table = comparison_table.pivot(
    index=['distance', 'offence'],
    columns='location',
    values=['schoolday_count', 'holiday_count', 'schoolday_rate', 'holiday_rate', 'rate_ratio']
)

In [ ]:
# reorder multiindex levels
comparison_table.columns = (
    comparison_table.columns.reorder_levels([1, 0])
)

In [ ]:
# arrange Within first, followed by Further than
comparison_table = comparison_table.reindex(
    columns=['Within', 'Further than'],
    level=0
)

In [ ]:
# return distance and offence to ordinary columns
comparison_table = comparison_table.reset_index()

In [ ]:
# flatten and rename the columns
comparison_table.columns = [
    'distance',
    'Crime type',
    'Within: term count',
    'Within: holiday count',
    'Within: term rate',
    'Within: holiday rate',
    'Within: rate ratio',
    'Further than: term count',
    'Further than: holiday count',
    'Further than: term rate',
    'Further than: holiday rate',
    'Further than: rate ratio'
]

In [ ]:
# compare the schoold day rate within buffer to its beyond counterpart
comparison_table['schoolday w2f ratio'] = (comparison_table['Within: term rate']
                                           / comparison_table['Further than: term rate'])

comparison_table['holiday w2f ratio'] = (comparison_table['Within: holiday rate']
                                         / comparison_table['Further than: holiday rate'])

In [ ]:
# add checks to the number of crimes
comparison_table['Total within'] = (comparison_table['Within: term count']
                                    + comparison_table['Within: holiday count'])

comparison_table['Total further'] = (comparison_table['Further than: term count']
                                     + comparison_table['Further than: holiday count'])

comparison_table['Total'] = (comparison_table['Total within']
                             + comparison_table['Total further'])

Plot the data

In [ ]:
# set the distances
distances = [250, 500, 750, 1000]

In [ ]:
# set the layout
fig, axes = plt.subplots(
    nrows=4, ncols=1,
    figsize=(12, 30), sharex=False)

# loop for each distance
for distance, ax in zip(distances, axes):
    plot_data = (
        comparison_table[comparison_table['distance'] == distance]
        .set_index('Crime type')[['schoolday w2f ratio', 'holiday w2f ratio']]
        .sort_values('schoolday w2f ratio', ascending=True))

    # plot the data
    plot_data.plot(kind='barh', ax=ax, width=0.8)

    # add baseline
    ax.axvline(x=1, color='black', linestyle='--', linewidth=1)

    # labels and titles
    ax.set_title(f'{distance} m: Within-to-Further rate ratio')
    ax.set_xlabel('Within-to-Further rate ratio')
    ax.set_ylabel('Crime type')
    ax.legend(title='Day type')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## 5.4 Does the school-day-to-holiday rate ratio change as the buffer increases?

In [ ]:
#get the unique offences
unique_offences = comparison_table['Crime type'].unique()

In [ ]:
# loop through each offence
for offence in unique_offences:

    offence_data = comparison_table[comparison_table['Crime type'] == offence]
    offence_data = offence_data.sort_values('distance')

    plt.figure(figsize=(15, 4.5))

    # within rate ratio
    plt.plot(offence_data['distance'], offence_data['Within: rate ratio'],
             marker='o', linewidth=2, color='#1f77b4', label='Within')

    # further-than rate ratio
    plt.plot(offence_data['distance'], offence_data['Further than: rate ratio'],
             marker='s', linewidth=2, color='#ff7f0e', linestyle='--', label='Further than')

    # reference value
    plt.axhline(y=1, color='black', linestyle=':', linewidth=1)

    plt.title(f'Distance Decay Profile for {offence.title()}', fontsize=12, fontweight='bold')

    plt.xlabel('Distance from school (metres)')
    plt.ylabel('School-day-to-holiday rate ratio')
    plt.xticks(offence_data['distance'])
    plt.grid(True, linestyle=':', alpha=0.4)
    plt.legend(frameon=True)
    plt.tight_layout()
    plt.show()

### By what percentage did the rate ratio change when moving from one buffer distance to the next?

In [ ]:
# loop through crime types
for offence in unique_offences:

    offence_data = (comparison_table[comparison_table['Crime type'] == offence]
                    .sort_values('distance')
                    .copy())

    # percentage change from the previous distance
    offence_data['within_rolling_pct'] = (offence_data['Within: rate ratio']
                                          .pct_change(fill_method=None) * 100)

    offence_data['further_rolling_pct'] = (offence_data['Further than: rate ratio']
                                           .pct_change(fill_method=None) * 100)

    # set the layout
    plt.figure(figsize=(12, 4.5))

    # plot the within
    plt.plot(offence_data['distance'].iloc[1:], offence_data['within_rolling_pct'].iloc[1:],
             marker='o', linewidth=2, label='Within')

    plt.plot(offence_data['distance'].iloc[1:], offence_data['further_rolling_pct'].iloc[1:],
             marker='s', linewidth=2, linestyle='--', label='Further than')

    plt.axhline(y=0, color='black', linestyle=':', alpha=0.5)

    plt.title(f'Percentage Change by Distance: {offence.title()}', fontweight='bold')
    plt.xlabel('New buffer distance (metres)')
    plt.ylabel('Change from previous distance (%)')
    plt.xticks(offence_data['distance'].iloc[1:])
    plt.grid(True, linestyle=':', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

## 5.5 Which of the crime types is associated school day?

In [ ]:
# calculate the ratio of what is happening within at school time to what is happening at holiday
comparison_table['w2f ratio of ratio'] = (comparison_table['schoolday w2f ratio']
                                          / comparison_table['holiday w2f ratio'])

In [ ]:
comparison_table.to_csv('euclidian_crimetype_rate_comparison_table_summary.csv')

In [ ]:
# set the distances
distances = [250, 500, 750, 1000]

In [ ]:
# initialise layout
fig, axes = plt.subplots(
    nrows=4, ncols=1,
    figsize=(16, 30), sharex=False)

# loop for each distance
for distance, ax in zip(distances, axes):
    plot_data = (
        comparison_table[comparison_table['distance'] == distance]
        .set_index('Crime type')['w2f ratio of ratio'])

    # exclude inf and sort
    plot_data = plot_data[np.isfinite(plot_data)].sort_values(ascending=True)

    # plot the data
    plot_data.plot(kind='barh', ax=ax, color='steelblue')

    ax.axvline(x=1, color='black', linestyle='--', linewidth=1)

    # labels and titles
    ax.set_title(f'{distance} m: School-day association by crime type')
    ax.set_xlabel('Ratio of W2F ratios')
    ax.set_ylabel('Crime type')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

At 250 meters, Miscellanous crimes against society, violence with injury, possession of weapons, shoplifting, theft, other theft, public order offences and robbery of personal property are indicative.

At 500 meters, the associative types include rape, possession of weapons, robbery of business property, robbery of personal perty, miscellaneous crimes against society, trafficiking in controlled drugs, violence with injury, shoplifting, public order offences.

At 750 meters, there are robbery of business property, trafficking in controlled drugs, robbery of personal property, other sexual offences, rape, public order offences, violence with injury.

At 1000 meters, it is now robbery of business property, rape, other sexual offences, trafficking in controlled drugs, other theft, and robbery of personal property.

## 5.6 Do the observed school-day and holiday crime counts differ from the counts expected based on the number of exposure days?

In [ ]:
# select the variables
crimes_test_df = comparison_table[
    ['distance', 'Crime type', 'Within: term count',
     'Within: holiday count', 'Further than: term count',
     'Further than: holiday count', 'Total']].copy()

In [ ]:
# calculate  total number of days
total_days = n_schooldays + n_holidays

# calculate proportions of each day type
schoolday_prop = n_schooldays / total_days
holiday_prop = n_holidays / total_days

In [ ]:
# calculate goodness of fit
results = []

# iterate the rows
for _, row in crimes_test_df.iterrows():
    # get observed crimes
    schoolday_within = row['Within: term count']
    holiday_within  = row['Within: holiday count']
    schoolday_further = row['Further than: term count']
    holiday_further  = row['Further than: holiday count']
    
    # calculate total volumes per spatial zone to protect against geographic area size differences
    total_within = schoolday_within + holiday_within
    total_further = schoolday_further + holiday_further
    
    # calculate expected counts
    schoolday_within_exp = total_within * schoolday_prop
    holiday_within_exp  = total_within * holiday_prop
    schoolday_further_exp = total_further * schoolday_prop
    holiday_further_exp  = total_further * holiday_prop

    # compute the ch12 test
    observed_cells = [schoolday_within, holiday_within, schoolday_further, holiday_further]
    expected_cells = [schoolday_within_exp, holiday_within_exp, schoolday_further_exp, holiday_further_exp]
    
    # run adjusted Chi-Square test
    # ddof=1 adjusts the degrees of freedom to exactly 2, since 2 localised totals are used
    chi2_stat, p_val = stats.chisquare(f_obs=observed_cells, f_exp=expected_cells, ddof=1)

    # append the values
    results.append({
        'Crime type': row['Crime type'],
        'distance': row['distance'],
        'Chi-square': chi2_stat,
        'p_value': p_val
    })

In [ ]:
crimes_chi_results = pd.DataFrame(results)

In [ ]:
# merge with teh selected group
crimes_test_df = crimes_test_df.merge(
    crimes_chi_results, 
    on=['Crime type', 'distance'], 
    how='left'
)

In [ ]:
# sort the p values
crimes_test_df = crimes_test_df.sort_values(by='p_value', ascending=True)
crimes_test_df[crimes_test_df['p_value'] < 0.05]

Temporal distribution differs from the exposure-based expectation in at least one spatial category for just three crime types: violence without injury, public order offences, and violenece with injury. Interestingly all three are are significant across all the buffer categories.

I will control the proportion of false discovery using the Benjamini–Hochberg test, adjusting the p-value

In [ ]:
# convert the results list into another DataFrame
results_df = pd.DataFrame(results)

In [ ]:
# adjust all p-values together
results_df['adjusted_p_value'] = multipletests(
    results_df['p_value'],
    method='fdr_bh'
)[1]

In [ ]:
results_df[results_df['adjusted_p_value'] < 0.05]

After the test, same three crime types were observed to show statistical signidficance. While public order offences and violence without injury were associated across the buffers, violence with injury is only associated at 250 meters.